# Experiments audit — provenance for the four resource-advantage experiments

Self-contained recomputation of every number in the paper's four experiment
subsections and the summary table, directly from the run outputs. **No imports from
`scripts/` or `cag`** — helpers are defined inline.

**How to run:** select the `.venv` kernel and *Run All*. Results are written to
`paper/tables/audit_experiments.csv`, `paper/tables/table1_experiments.csv` (the paper's
Table 1), and `paper/tables/audit_experiments_log.txt`. Read those files instead of the
notebook.

**Sections (research title — internal tag):**
1. Reach advantage — *Bite 1* (recomputed from the Tier-1 reach runs; the dedicated
   reach-ladder runs are pending)
2. Frequency advantage — *Bite 2*
3. Targeting a limited broadcast — *Bite 3* (a null)
4. Breadth vs depth at a fixed impression budget — *Bite 4*

Outcome metric: within-person difference in the final **package index** (−3…+3), paired
per agent within seed, pooled over the three seeds (42/43/44).


In [1]:
# ---------------------------------------------------------------------------
# Setup & inline helpers (self-contained; no imports from scripts/ or cag)
# ---------------------------------------------------------------------------
from pathlib import Path
import glob
import numpy as np
import pandas as pd
from scipy import stats

REPO = Path.cwd()
while not (REPO / "data" / "output").exists() and REPO != REPO.parent:
    REPO = REPO.parent
EXP = REPO / "data" / "output" / "experiments"        # Tier-1 reach runs
ASYM = REPO / "data" / "output" / "asymmetry_tests"   # draft-45 experiments
TABLES = REPO / "paper" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)
print("REPO :", REPO)
print("ASYM :", ASYM, "(exists:", ASYM.exists(), ")")
print("EXP  :", EXP, "(exists:", EXP.exists(), ")")

LOG = []
def out(msg=""):
    print(msg)
    LOG.append(str(msg))

PROV = []
def check(claim, paper, computed, source, tol=0.02):
    ok = computed is not None and abs(float(computed) - float(paper)) <= tol
    status = "PASS" if ok else "CHECK"
    PROV.append({"claim": claim, "paper": paper,
                 "computed": None if computed is None else round(float(computed), 4),
                 "source_runs": source, "status": status})
    c = "n/a" if computed is None else f"{round(float(computed),4):+.4f}"
    out(f"  [{status}] {claim}")
    out(f"         paper={paper}   computed={c}   <- {source}")
    return ok

def latest_subdir(run_dir):
    subs = sorted([p for p in Path(run_dir).iterdir() if p.is_dir()])
    return subs[-1] if subs else Path(run_dir)

def find_run(base, token, seed):
    hits = sorted(glob.glob(str(Path(base) / f"run_*_{token}_s{seed}")))
    return hits[0] if hits else None

def load_endpoints(run_dir):
    d = latest_subdir(run_dir)
    pkg = pd.read_csv(d / "package_index_trajectories.csv")
    gt = pd.read_csv(d / "package_ground_truth.csv").set_index("agent_id")["ground_truth"]
    attr = pd.read_csv(d / "agent_attributes.csv").set_index("agent_id")["political_exposure"]
    dN = sorted(pkg["day"].unique())[-1]
    end = pkg[pkg.day == dN].set_index("agent_id")["package_index"]
    return pd.DataFrame({"end": end, "gt": gt, "bucket": attr}).dropna().reset_index()

def mean_ci(d):
    d = np.asarray(d, float)
    m = d.mean()
    h = d.std(ddof=1) / np.sqrt(len(d)) * stats.t.ppf(0.975, len(d) - 1)
    return m, m - h, m + h

def pooled_diff(base, tok_pro, tok_scep, seeds=(42, 43, 44), bucket=None):
    chunks, per_seed, srcs = [], {}, []
    for s in seeds:
        rp, rs = find_run(base, tok_pro, s), find_run(base, tok_scep, s)
        if rp is None or rs is None:
            per_seed[s] = None
            continue
        srcs += [Path(rp).name, Path(rs).name]
        a, b = load_endpoints(rp), load_endpoints(rs)
        if bucket:
            a, b = a[a.bucket == bucket], b[b.bucket == bucket]
        m = a[["agent_id", "end", "gt"]].merge(b[["agent_id", "end"]],
                                               on="agent_id", suffixes=("_p", "_s"))
        d = (m["end_p"] - m["end_s"]).to_numpy()
        chunks.append(d)
        per_seed[s] = float(np.mean(d))
    if not chunks:
        return None
    alld = np.concatenate(chunks)
    m, lo, hi = mean_ci(alld)
    _, p = stats.ttest_1samp(alld, 0.0)
    signs = {np.sign(v) for v in per_seed.values() if v is not None}
    return {"n": len(alld), "mean": float(m), "lo": float(lo), "hi": float(hi), "p": float(p),
            "per_seed": {k: (None if v is None else round(v, 3)) for k, v in per_seed.items()},
            "same_sign": (len(signs) == 1), "src": sorted(set(srcs))}

def support_share(base, token, seeds=(42, 43, 44)):
    """Mean support share (% end > 0) across the runs for a condition token."""
    vals = []
    for s in seeds:
        r = find_run(base, token, s)
        if r is None:
            continue
        e = load_endpoints(r)["end"]
        vals.append((e > 0).mean() * 100)
    return float(np.mean(vals)) if vals else None

out("helpers ready")


REPO : /Users/ajaykumar/src/GitHub/Climate-Action-GABM
ASYM : /Users/ajaykumar/src/GitHub/Climate-Action-GABM/data/output/asymmetry_tests (exists: True )
EXP  : /Users/ajaykumar/src/GitHub/Climate-Action-GABM/data/output/experiments (exists: True )
helpers ready


In [2]:
# ===========================================================================
# EXPERIMENT 1 - Reach advantage (internal: Bite 1)
#   Recomputed from the Tier-1 reach runs (data/output/experiments); the dedicated
#   draft-45 reach-ladder runs are pending and will replace these.
# ===========================================================================
RES = {}   # collect per-experiment headline for Table 1
out("")
out("=" * 74)
out("EXPERIMENT 1 - Reach advantage (internal: Bite 1; from Tier-1 reach runs)")
out("=" * 74)

reach = pooled_diff(EXP, "tier1_green_dom", "tier1_reform_dom")
reach_both = pooled_diff(EXP, "tier1_green_dom", "tier1_reform_dom", bucket="both")
check("Reach: whole-population effect (points)",
      0.42, None if reach is None else reach["mean"],
      "" if reach is None else ",".join(reach["src"]), tol=0.03)
check("Reach: among those who hear both sides",
      0.50, None if reach_both is None else reach_both["mean"], "tier1 both-bucket", tol=0.04)
if reach:
    out(f"         95% CI [{reach['lo']:+.3f}, {reach['hi']:+.3f}]  same sign every seed: "
        f"{reach['same_sign']}  per-seed={reach['per_seed']}")
RES["Reach"] = {
    "buys": "reach more citizens",
    "whole": None if reach is None else f"{reach['mean']:+.2f} [{reach['lo']:+.2f}, {reach['hi']:+.2f}]",
    "both": None if reach_both is None else f"{reach_both['mean']:+.2f}",
    "reliability": None if reach is None else ("same sign 3/3 seeds" if reach["same_sign"] else "mixed sign"),
    "n_runs": len(glob.glob(str(EXP / "run_*_tier1_*_s*"))),
    "note": "validated backbone (Tier 1-3); graded ladder pending",
}



EXPERIMENT 1 - Reach advantage (internal: Bite 1; from Tier-1 reach runs)
  [PASS] Reach: whole-population effect (points)
         paper=0.42   computed=+0.4228   <- run_6458327_tier1_green_dom_s42,run_6458328_tier1_green_dom_s43,run_6458329_tier1_green_dom_s44,run_6458330_tier1_reform_dom_s42,run_6458331_tier1_reform_dom_s43,run_6458332_tier1_reform_dom_s44
  [PASS] Reach: among those who hear both sides
         paper=0.5   computed=+0.4981   <- tier1 both-bucket
         95% CI [+0.337, +0.508]  same sign every seed: True  per-seed={42: 0.532, 43: 0.423, 44: 0.313}


In [3]:
# ===========================================================================
# EXPERIMENT 2 - Frequency advantage (internal: Bite 2)
#   SBM, full reach both sides; lever = broadcasts per day. Contrast = louder-pro
#   minus louder-sceptic at matched ratio, paired within seed, pooled n=300.
# ===========================================================================
out("")
out("=" * 74)
out("EXPERIMENT 2 - Frequency advantage (internal: Bite 2)")
out("=" * 74)

f21 = pooled_diff(ASYM, "d45_freq_green2v1", "d45_freq_reform1v2")
f31 = pooled_diff(ASYM, "d45_freq_green3v1", "d45_freq_reform1v3")
check("Frequency 2:1 - whole population (points)",
      0.158, None if f21 is None else f21["mean"], "" if f21 is None else ",".join(f21["src"]), tol=0.03)
check("Frequency 3:1 - whole population (points)",
      0.277, None if f31 is None else f31["mean"], "" if f31 is None else ",".join(f31["src"]), tol=0.03)
f21b = pooled_diff(ASYM, "d45_freq_green2v1", "d45_freq_reform1v2", bucket="both")
f31b = pooled_diff(ASYM, "d45_freq_green3v1", "d45_freq_reform1v3", bucket="both")
check("Frequency 2:1 - among those who hear both sides",
      0.146, None if f21b is None else f21b["mean"], "freq both-bucket", tol=0.04)
check("Frequency 3:1 - among those who hear both sides",
      0.338, None if f31b is None else f31b["mean"], "freq both-bucket", tol=0.05)

sup_lo = support_share(ASYM, "d45_freq_reform1v3")   # sceptic loudest
sup_hi = support_share(ASYM, "d45_freq_green3v1")    # pro loudest
check("Frequency: support share when sceptic side loudest (%)",
      77.7, sup_lo, "d45_freq_reform1v3", tol=3.0)
check("Frequency: support share when pro side loudest (%)",
      83.3, sup_hi, "d45_freq_green3v1", tol=3.0)
if f31:
    out(f"         dose-response 2:1 -> 3:1 : {f21['mean']:+.3f} -> {f31['mean']:+.3f}  "
        f"(same sign every seed: {f31['same_sign']})")
RES["Frequency"] = {
    "buys": "broadcast more often",
    "whole": (None if f21 is None or f31 is None else
              f"2:1 {f21['mean']:+.2f} [{f21['lo']:+.2f}, {f21['hi']:+.2f}]; "
              f"3:1 {f31['mean']:+.2f} [{f31['lo']:+.2f}, {f31['hi']:+.2f}]"),
    "both": (None if f21b is None or f31b is None else f"{f21b['mean']:+.2f} / {f31b['mean']:+.2f}"),
    "reliability": None if f31 is None else ("same sign 3/3 seeds" if f31["same_sign"] else "mixed sign"),
    "n_runs": len(glob.glob(str(ASYM / "run_*_d45_freq_*_s*"))),
    "note": "dose-response; weaker than reach",
}



EXPERIMENT 2 - Frequency advantage (internal: Bite 2)
  [PASS] Frequency 2:1 - whole population (points)
         paper=0.158   computed=+0.1578   <- run_6515406_d45_freq_green2v1_s42,run_6515407_d45_freq_green2v1_s43,run_6515408_d45_freq_green2v1_s44,run_6515412_d45_freq_reform1v2_s42,run_6515413_d45_freq_reform1v2_s43,run_6515414_d45_freq_reform1v2_s44
  [PASS] Frequency 3:1 - whole population (points)
         paper=0.277   computed=+0.2772   <- run_6515409_d45_freq_green3v1_s42,run_6515410_d45_freq_green3v1_s43,run_6515411_d45_freq_green3v1_s44,run_6515415_d45_freq_reform1v3_s42,run_6515416_d45_freq_reform1v3_s43,run_6515417_d45_freq_reform1v3_s44
  [PASS] Frequency 2:1 - among those who hear both sides
         paper=0.146   computed=+0.1463   <- freq both-bucket
  [PASS] Frequency 3:1 - among those who hear both sides
         paper=0.338   computed=+0.3380   <- freq both-bucket
  [PASS] Frequency: support share when sceptic side loudest (%)
         paper=77.7   computed=+77.66

In [4]:
# ===========================================================================
# EXPERIMENT 3 - Targeting a limited broadcast (internal: Bite 3) - a NULL
#   BA hub network; pro side throttled to 25% reach. Lever = which quarter it keeps
#   (random / persuadable / degree). Contrast = mode - random, paired within seed.
# ===========================================================================
out("")
out("=" * 74)
out("EXPERIMENT 3 - Targeting a limited broadcast (internal: Bite 3; a null)")
out("=" * 74)

tp = pooled_diff(ASYM, "d45_tgt_persuadable", "d45_tgt_random")
td = pooled_diff(ASYM, "d45_tgt_degree", "d45_tgt_random")
check("Targeting: persuadable - random (points; expect ~0, n.s.)",
      0.044, None if tp is None else tp["mean"], "" if tp is None else ",".join(tp["src"]), tol=0.03)
check("Targeting: degree(hubs) - random (points; expect ~0, n.s.)",
      0.027, None if td is None else td["mean"], "" if td is None else ",".join(td["src"]), tol=0.03)
if tp and td:
    out(f"         p-values: persuadable p={tp['p']:.2f}, degree p={td['p']:.2f} "
        f"(both non-significant)")
    out(f"         sign across seeds: persuadable {tp['per_seed']}  degree {td['per_seed']}")
    out("         -> point estimates weakly favour smart targeting but are neither")
    out("            significant nor sign-stable: a genuine null, and a discrimination")
    out("            check the instrument passes (not every lever registers).")
RES["Targeting"] = {
    "buys": "aim a limited broadcast",
    "whole": (None if tp is None or td is None else
              f"persuadable {tp['mean']:+.2f} (p={tp['p']:.2f}); degree {td['mean']:+.2f} (p={td['p']:.2f})"),
    "both": "~0 (n.s.)",
    "reliability": "mixed sign across seeds",
    "n_runs": len(glob.glob(str(ASYM / "run_*_d45_tgt_*_s*"))),
    "note": "null / boundary condition",
}



EXPERIMENT 3 - Targeting a limited broadcast (internal: Bite 3; a null)
  [PASS] Targeting: persuadable - random (points; expect ~0, n.s.)
         paper=0.044   computed=+0.0439   <- run_6535130_d45_tgt_random_s42,run_6535131_d45_tgt_random_s43,run_6535132_d45_tgt_random_s44,run_6535133_d45_tgt_persuadable_s42,run_6535134_d45_tgt_persuadable_s43,run_6535135_d45_tgt_persuadable_s44
  [PASS] Targeting: degree(hubs) - random (points; expect ~0, n.s.)
         paper=0.027   computed=+0.0267   <- run_6535130_d45_tgt_random_s42,run_6535131_d45_tgt_random_s43,run_6535132_d45_tgt_random_s44,run_6535136_d45_tgt_degree_s42,run_6535137_d45_tgt_degree_s43,run_6535138_d45_tgt_degree_s44
         p-values: persuadable p=0.23, degree p=0.52 (both non-significant)
         sign across seeds: persuadable {42: 0.075, 43: 0.063, 44: -0.007}  degree {42: 0.11, 43: -0.06, 44: 0.03}
         -> point estimates weakly favour smart targeting but are neither
            significant nor sign-stable: a genuine

In [5]:
# ===========================================================================
# EXPERIMENT 4 - Breadth vs depth at a fixed impression budget (internal: Bite 4)
#   Matched total impressions spent broad+shallow (depth2) vs narrow+deep (depth4).
#   Contrast = depth2 - depth4; > 0 => breadth (reach) beats depth (frequency).
# ===========================================================================
out("")
out("=" * 74)
out("EXPERIMENT 4 - Breadth vs depth at equal impressions (internal: Bite 4)")
out("=" * 74)

iso = pooled_diff(ASYM, "d45_iso_depth2", "d45_iso_depth4")
isob = pooled_diff(ASYM, "d45_iso_depth2", "d45_iso_depth4", bucket="both")
check("Breadth vs depth: whole population (points; > 0 => breadth wins)",
      0.120, None if iso is None else iso["mean"], "" if iso is None else ",".join(iso["src"]), tol=0.03)
check("Breadth vs depth: among those who hear both sides",
      0.203, None if isob is None else isob["mean"], "iso both-bucket", tol=0.05)
if iso:
    out(f"         95% CI [{iso['lo']:+.3f}, {iso['hi']:+.3f}]  p={iso['p']:.2e}  "
        f"same sign every seed: {iso['same_sign']}")
    out("         -> total impressions is NOT a sufficient statistic: spreading a fixed")
    out("            budget wide beats stacking it deep. Reach and frequency are not")
    out("            interchangeable.")
RES["Iso-impression"] = {
    "buys": "breadth vs depth (equal impressions)",
    "whole": None if iso is None else f"{iso['mean']:+.2f} [{iso['lo']:+.2f}, {iso['hi']:+.2f}]",
    "both": None if isob is None else f"{isob['mean']:+.2f}",
    "reliability": None if iso is None else ("same sign 3/3 seeds" if iso["same_sign"] else "mixed sign"),
    "n_runs": len(glob.glob(str(ASYM / "run_*_d45_iso_*_s*"))),
    "note": "breadth beats depth (small margin)",
}



EXPERIMENT 4 - Breadth vs depth at equal impressions (internal: Bite 4)
  [PASS] Breadth vs depth: whole population (points; > 0 => breadth wins)
         paper=0.12   computed=+0.1200   <- run_6535139_d45_iso_depth2_s42,run_6535140_d45_iso_depth2_s43,run_6535141_d45_iso_depth2_s44,run_6535142_d45_iso_depth4_s42,run_6535143_d45_iso_depth4_s43,run_6535144_d45_iso_depth4_s44
  [PASS] Breadth vs depth: among those who hear both sides
         paper=0.203   computed=+0.2028   <- iso both-bucket
         95% CI [+0.031, +0.209]  p=8.18e-03  same sign every seed: True
         -> total impressions is NOT a sufficient statistic: spreading a fixed
            budget wide beats stacking it deep. Reach and frequency are not
            interchangeable.


In [6]:
# ===========================================================================
# Table 1 (paper), program tally, and export
# ===========================================================================
out("")
out("=" * 74)
out("TABLE 1 - the four experiments")
out("=" * 74)
order = ["Reach", "Frequency", "Targeting", "Iso-impression"]
t1 = pd.DataFrame([{
    "Experiment": k,
    "What the advantage buys": RES[k]["buys"],
    "Whole-population effect (95% CI)": RES[k]["whole"],
    "Among those who hear both sides": RES[k]["both"],
    "Reliability": RES[k]["reliability"],
    "n runs": RES[k]["n_runs"],
    "note": RES[k]["note"],
} for k in order if k in RES])
t1_path = TABLES / "table1_experiments.csv"
t1.to_csv(t1_path, index=False)
out(t1.to_string(index=False))

# Program tally (breadth sentence): count landed simulations on disk.
out("")
out("=" * 74)
out("PROGRAM TALLY - landed simulations on disk")
out("=" * 74)
counts = {
    "Tier P (persona)": len([r for r in range(6457850, 6457859)
                             if (EXP / f"run_{r}").exists()]),
    "Tier 1 (reach)": len(glob.glob(str(EXP / "run_*_tier1_*_s*"))),
    "Tier 3 network (BA+WS)": (len(glob.glob(str(EXP / "run_*_tier3_ba_*_s*")))
                               + len(glob.glob(str(EXP / "run_*_tier3_ws_*_s*")))),
    "Tier 3 model (Llama, seed 42)": (len(glob.glob(str(EXP / "run_6475093*")))
                                      + len(glob.glob(str(EXP / "run_6479341*")))
                                      + len(glob.glob(str(EXP / "run_6479390*")))),
    "draft-45 experiments (freq+tgt+iso)": len(glob.glob(str(ASYM / "run_*_d45_*_s*"))),
}
for k, v in counts.items():
    out(f"   {k:38s} {v:3d}")
total = sum(counts.values())
out(f"   {'TOTAL landed':38s} {total:3d}")
out("   (Tier 3 SBM arm reuses the Tier-1 runs; the dedicated draft-45 reach-ladder")
out("    runs are pending and not counted here.)")

# Export the audit trail.
prov = pd.DataFrame(PROV, columns=["claim", "paper", "computed", "source_runs", "status"])
csv_path = TABLES / "audit_experiments.csv"
log_path = TABLES / "audit_experiments_log.txt"
prov.to_csv(csv_path, index=False)
n_pass = int((prov.status == "PASS").sum())
n_check = int((prov.status == "CHECK").sum())
out("")
out(f"SUMMARY  PASS={n_pass}  CHECK={n_check}   (CHECK rows need a look)")
log_path.write_text("\n".join(LOG))
print("WROTE", t1_path)
print("WROTE", csv_path)
print("WROTE", log_path)
prov



TABLE 1 - the four experiments
    Experiment              What the advantage buys                   Whole-population effect (95% CI) Among those who hear both sides             Reliability  n runs                                                 note
         Reach                  reach more citizens                               +0.42 [+0.34, +0.51]                           +0.50     same sign 3/3 seeds      12 validated backbone (Tier 1-3); graded ladder pending
     Frequency                 broadcast more often 2:1 +0.16 [+0.08, +0.23]; 3:1 +0.28 [+0.20, +0.36]                   +0.15 / +0.34     same sign 3/3 seeds      12                     dose-response; weaker than reach
     Targeting              aim a limited broadcast  persuadable +0.04 (p=0.23); degree +0.03 (p=0.52)                       ~0 (n.s.) mixed sign across seeds       9                            null / boundary condition
Iso-impression breadth vs depth (equal impressions)                               +0.12 

,claim,paper,computed,source_runs,status
0,Reach: whole-population effect (points),0.420,0.4228,"run_6458327_tier1_green_dom_s42,run_6458328_ti...",PASS
1,Reach: among those who hear both sides,0.500,0.4981,tier1 both-bucket,PASS
2,Frequency 2:1 - whole population (points),0.158,0.1578,"run_6515406_d45_freq_green2v1_s42,run_6515407_...",PASS
3,Frequency 3:1 - whole population (points),0.277,0.2772,"run_6515409_d45_freq_green3v1_s42,run_6515410_...",PASS
4,Frequency 2:1 - among those who hear both sides,0.146,0.1463,freq both-bucket,PASS
5,Frequency 3:1 - among those who hear both sides,0.338,0.3380,freq both-bucket,PASS
6,Frequency: support share when sceptic side lou...,77.700,77.6667,d45_freq_reform1v3,PASS
7,Frequency: support share when pro side loudest...,83.300,83.3333,d45_freq_green3v1,PASS
8,Targeting: persuadable - random (points; expec...,0.044,0.0439,"run_6535130_d45_tgt_random_s42,run_6535131_d45...",PASS
9,Targeting: degree(hubs) - random (points; expe...,0.027,0.0267,"run_6535130_d45_tgt_random_s42,run_6535131_d45...",PASS
